In [6]:
import sys
import os

sys.path.append("/home/datalab/nfs/deepfm/")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pandas as pd
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import json
from tqdm import tqdm

from src.datasets import StreamDataset
from src.datasets.collate import collate_fn
from src.models import DeepFM

## Preparing Model

In [7]:
kt_features = [    
    "inn_kt_index",
    "okved_cd_kt_index",
    "okato_cd_kt_index",
    "bic_kt_34_index",
    "bic_kt_56_index",
    "bic_kt_79_index",
    "num_kt_13_index",
    "num_kt_45_index",
    "num_kt_68_index",
    "okved_cd_kt_lvl1_index",
    "okved_cd_kt_lvl2_index",
    "okved_cd_kt_lvl3_index"]

kt_double_features = [
    "kt_avg_sum",
    "kt_stddev_sum",
    "kt_min_sum",
    "kt_max_sum",
    "kt_median_sum",
    "kt_skewness_sum",
    "kt_buyers_count"]

dt_features =  [    
    "inn_dt_index",
    "okved_cd_dt_index",
    "okato_cd_dt_index",
    "bic_dt_34_index",
    "bic_dt_56_index",
    "bic_dt_79_index",
    "num_dt_13_index",
    "num_dt_45_index",
    "num_dt_68_index",
    "okved_cd_dt_lvl1_index",
    "okved_cd_dt_lvl2_index",
    "okved_cd_dt_lvl3_index"]

dt_double_features = [
    "dt_avg_sum",
    "dt_stddev_sum",
    "dt_min_sum",
    "dt_max_sum",
    "dt_median_sum",
    "dt_skewness_sum",
    "dt_buyers_count"]

label_column = "label"

In [8]:
device = torch.device("cuda")

with open("/home/datalab/nfs/deepfm/data/train_21/user_feature_sizes.json", "r") as f:
    user_feature_sizes = json.load(f)

with open("/home/datalab/nfs/deepfm/data/train_21/item_feature_sizes.json", "r") as f:
    item_feature_sizes = json.load(f)

model = DeepFM(
    embed_dim=128,
    num_user_double_feats=7,
    num_item_double_feats=7,
    user_feature_sizes=user_feature_sizes,
    item_feature_sizes=item_feature_sizes,
).to(device)

checkpoint = torch.load("/home/datalab/nfs/deepfm/deepfm_logs/train_21_boevoy_zapusk_vse_fichi/model_best.pth", device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

DeepFM(
  (user_embed): FeatureEmbedding(
    (embeddings): ModuleDict(
      (inn_dt_index): Embedding(2798745, 128)
      (okved_cd_dt_index): Embedding(2643, 51)
      (okato_cd_dt_index): Embedding(60671, 128)
      (bic_dt_34_index): Embedding(84, 9)
      (bic_dt_56_index): Embedding(56, 7)
      (bic_dt_79_index): Embedding(96, 9)
      (num_dt_13_index): Embedding(14, 3)
      (num_dt_45_index): Embedding(26, 5)
      (num_dt_68_index): Embedding(4, 2)
      (okved_cd_dt_lvl1_index): Embedding(91, 9)
      (okved_cd_dt_lvl2_index): Embedding(101, 10)
      (okved_cd_dt_lvl3_index): Embedding(69, 8)
    )
  )
  (item_embed): FeatureEmbedding(
    (embeddings): ModuleDict(
      (inn_kt_index): Embedding(2364265, 128)
      (okved_cd_kt_index): Embedding(2580, 50)
      (okato_cd_kt_index): Embedding(59481, 128)
      (bic_kt_34_index): Embedding(79, 8)
      (bic_kt_56_index): Embedding(51, 7)
      (bic_kt_79_index): Embedding(96, 9)
      (num_kt_13_index): Embedding(17, 4)
  

## Preparing indices

In [9]:
joined_recs = pd.read_excel("/home/datalab/nfs/deepfm/data/test_07_08_2025/joined_andrey_recs_dt_4testing_07_08.xlsx")

import pyarrow.parquet as pq
import pandas as pd
import gzip
import pickle

kt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_kt/data"
labels_df = pq.read_table(kt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_kt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

dt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_dt/data"
labels_df = pq.read_table(dt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_dt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

Importing features

In [10]:
with gzip.open("../data/train_21/dt_features_dict.pkl.gz", "rb") as f: dt_feat = pickle.load(f)
with gzip.open("../data/train_21/kt_features_dict.pkl.gz", "rb") as f: kt_feat = pickle.load(f)

In [35]:
with gzip.open("../data/train_21/dt_embeddings_dict.pkl.gz", "rb") as f: dt_embs = pickle.load(f)
with gzip.open("../data/train_21/kt_embeddings_dict.pkl.gz", "rb") as f: kt_embs = pickle.load(f)

In [ ]:
joined_recs.head(2)

In [46]:
kt_attentions = {}
count = 0

for i, row in joined_recs.iterrows():
    inn_dt = row['nn_DT']
    inn_kt = row['nn_KT']
    
    if str(inn_kt) in inn_kt_to_index:
        iid = inn_kt_to_index.get(str(inn_kt), -1)
        data = kt_feat.get(iid, {})
        cat = torch.tensor([[data.get(f, 0) for f in kt_features]], dtype=torch.long, device="cuda")
        cont = torch.tensor([[data.get(f, 0.0) for f in kt_double_features]], dtype=torch.float32, device="cuda")
        emb0 = torch.tensor(kt_embs.get(iid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)
        with torch.no_grad():
            emb, attention = model.embed_item(cat, cont, emb0)
        # dt_embeddings[iid] = emb.squeeze(0).cpu().numpy()
        kt_attentions[inn_kt] = attention.squeeze(0).cpu().numpy()
    else:
        # print(inn_kt)
        count += 1


# dt_embeddings = {}
# kt_attentions = {}
# cold_recs = set()


In [58]:
importances_names = [
    "id_в_модели",
    "kt_ОКВЭД_важность",
    "kt_ОКАТО_важность",
    "kt_регион_важность",
    "kt_номер_ЦБ",
    "kt_номер_организации_в_ЦБ",
    "kt_номер_счёта",
    "kt_номер_счёта_2",
    "kt_валюта",
    "kt_оквэд_уровень_1",
    "kt_оквэд_уровень_2",
    "kt_овкэд_уровень_3",
    "kt_статистические_фичи",
    "kt_эмбеддинг_постновой"
]

In [59]:
for i, row in joined_recs.iterrows():
    inn_kt = row['nn_KT']
    if inn_kt in kt_attentions:
        for feature_name, importance in zip(importances_names, kt_attentions[inn_kt]):
            joined_recs.loc[i, feature_name] = importance

In [62]:
joined_recs.to_excel("../data/test_07_08_2025/andrey_recs_dt_with_attentions_26_08.xlsx", index=False)

In [63]:
joined_recs = pd.read_excel("/home/datalab/nfs/deepfm/data/test_07_08_2025/andrey_recs_dt_with_attentions_26_08.xlsx")

In [ ]:
joined_recs.head(5)